# Basic CNN Model


In [1]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics

class StreamlinedCNN(pl.LightningModule):
    def __init__(self, classes=10):
        super(StreamlinedCNN, self).__init__()
        # Initialize convolutional layers with batch normalization and pooling
        self.conv_layer1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.conv_layer2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv_layer3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pooling_layer = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc_layer1 = nn.Linear(64 * 8 * 8, 128)
        self.fc_layer2 = nn.Linear(128, classes)
        self.normalization_layer1 = nn.BatchNorm2d(16)
        self.normalization_layer2 = nn.BatchNorm2d(32)
        self.normalization_layer3 = nn.BatchNorm2d(64)
        self.regularization_layer = nn.Dropout(0.25)
        self.metric_accuracy = torchmetrics.Accuracy(num_classes=classes, average="macro", task="multiclass")

    def forward(self, inputs):
        inputs = self.pooling_layer(F.relu(self.normalization_layer1(self.conv_layer1(inputs))))
        inputs = self.pooling_layer(F.relu(self.normalization_layer2(self.conv_layer2(inputs))))
        inputs = self.pooling_layer(F.relu(self.normalization_layer3(self.conv_layer3(inputs))))
        inputs = torch.flatten(inputs, 1)
        inputs = F.relu(self.fc_layer1(inputs))
        inputs = self.regularization_layer(inputs)
        outputs = self.fc_layer2(inputs)
        return outputs

    def training_step(self, batch, batch_idx):
        data, labels = batch
        predictions = self(data)
        loss = F.cross_entropy(predictions, labels)
        self.log("training_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        data, labels = batch
        predictions = self(data)
        validation_loss = F.cross_entropy(predictions, labels)
        self.metric_accuracy(predictions, labels)
        self.log("validation_loss", validation_loss, on_epoch=True, prog_bar=True)
        self.log("validation_accuracy", self.metric_accuracy, on_epoch=True, prog_bar=True)

    def test_step(self, batch, batch_idx):
        data, labels = batch
        predictions = self(data)
        test_loss = F.cross_entropy(predictions, labels)
        self.metric_accuracy(predictions, labels)
        self.log("test_accuracy", self.metric_accuracy)
        self.log("test_loss", test_loss)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)  # Using a slightly adjusted learning rate



The Imagenette dataset is a smaller subset of 10 easily classified classes from Imagenet. It is available to download from `torchvision`, as shown in the cell below. There are 3 different sizes of the images available. Feel free to use whichever version you prefer. It might make a difference in the performance of your model.

**Note: After downloading the Imagenette dataset, you will need to set `download=False` in the cell below to avoid errors.**

In [3]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from torchvision import transforms
from torchvision.datasets import Imagenette
import torch
from torch.utils.data import DataLoader, random_split

# Setup transformations for data preprocessing
training_transformations = transforms.Compose([
    transforms.CenterCrop(160),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    transforms.Grayscale()  # Convert images to grayscale for uniformity
])

validation_transformations = transforms.Compose([
    transforms.CenterCrop(160),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    transforms.Grayscale()  # Maintain the same transformations for validation
])

# Load training data from Imagenette dataset
imagenette_train = Imagenette("../data/imagenette/train/", split="train", size="160px", download=False, transform=training_transformations)

# Define the sizes for splitting the dataset into training and validation
size_of_training = int(len(imagenette_train) * 0.9)
size_of_validation = len(imagenette_train) - size_of_training

# Set seed for reproducibility when splitting dataset
dataset_seed = torch.Generator().manual_seed(42)
imagenette_train, imagenette_validation = random_split(imagenette_train, [size_of_training, size_of_validation], generator=dataset_seed)
imagenette_validation.dataset.transform = validation_transformations

# Configure DataLoader for handling the dataset
training_loader = DataLoader(
    imagenette_train, batch_size=128, num_workers=8, shuffle=True, persistent_workers=True
)
validation_loader = DataLoader(
    imagenette_validation, batch_size=128, num_workers=8, shuffle=False, persistent_workers=True
)

# Setup the dataset for testing purposes
imagenette_test = Imagenette("../data/imagenette/test/", split="val", size="160px", download=False, transform=validation_transformations)

# Initialize the model
cnn_model = StreamlinedCNN()

# Setup EarlyStopping to monitor validation loss
early_stopping_handler = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5  # Number of epochs with no improvement after which training is stopped
)

# Setup model checkpointing to save the best model based on validation loss
checkpoint_saver = ModelCheckpoint(
    monitor="val_loss",
    mode="min"
)


In [8]:
# Fit the model
torch.set_float32_matmul_precision("high")
trainer = pl.Trainer(callbacks=[early_stopping_handler, checkpoint_saver], max_epochs=-1)

trainer.fit(model=cnn_model, train_dataloaders=training_loader, val_dataloaders=validation_loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

   | Name                 | Type               | Params | Mode 
---------------------------------------------------------------------
0  | conv_layer1          | Conv2d             | 160    | train
1  | conv_layer2          | Conv2d             | 4.6 K  | train
2  | conv_layer3          | Conv2d             | 18.5 K | train
3  | pooling_layer        | MaxPool2d          | 0      | train
4  | fc_layer1            | Linear             | 524 K  | train
5  | fc_layer2            | Linear             | 1.3 K  | train
6  | normalization_layer1 | BatchNorm2d        | 32     | train
7  | normalization_layer2 | BatchNorm2d        | 64     | train
8  | normalization_layer3 | BatchNorm2d        | 128    | train
9  | regularization_layer | Dropout            | 0      | train
10 | metric_accuracy      | MulticlassAccuracy | 0      | train
----------------------------------------------

Sanity Checking: |                                        | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/__init__.py", line 1694, in <module>
    _C._initExtension(_manager_path())
  File "<frozen importlib._bootstrap>", line 216, in _lock_unlock_module
KeyboardInterrupt


NameError: name 'exit' is not defined

In [9]:
# Evaluate the model on the test set
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=256, num_workers=8, shuffle=False, persistent_workers=True
)
trainer.test(model=model, dataloaders=test_loader)

NameError: name 'test_dataset' is not defined